In [11]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import yaml

ROOT = Path("..")
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

CRS_GEO = config["crs"]["geographic"]
CRS_AREA = config["crs"]["equal_area"]

sites = gpd.read_parquet(INTERIM / "osm_datacenters.parquet")
basins = gpd.read_parquet(INTERIM / "aqueduct_europe.parquet")

print(len(sites), "sites |", sites.crs)
print(len(basins), "basins |", basins.crs)

1124 sites | {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope"

In [12]:
# Use the site's centroid (lon/lat computed in step 1) as its location
site_points = gpd.GeoDataFrame(
    sites.drop(columns="geometry"),
    geometry=gpd.points_from_xy(sites["lon"], sites["lat"]),
    crs=CRS_GEO,
)
site_points.geometry.geom_type.value_counts()

Point    1124
Name: count, dtype: int64

In [13]:
joined = gpd.sjoin(site_points, basins, how="left", predicate="within")

matched = joined["pfaf_id"].notna().sum()
unmatched = joined["pfaf_id"].isna().sum()
duplicates = joined["campus_id"].duplicated().sum()

print(f"Matched: {matched} | Outside every basin: {unmatched} | Duplicated sites: {duplicates}")

Matched: 1120 | Outside every basin: 4 | Duplicated sites: 0


In [14]:
outside = joined[joined["pfaf_id"].isna()]
outside[["campus_id", "country", "name", "operators", "lat", "lon"]]

,campus_id,country,name,operators,lat,lon
0,existing_0,ES,Aire Canarias,AIRE,28.118717,-15.429130
317,existing_31,NL,Bluenap Americas,,12.177858,-68.955208
1073,existing_991,NO,R74n Vault,,77.475486,23.475382
1074,existing_992,FR,Data,ITH Center,-12.778454,45.225200


In [15]:
with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)
MAX_KM = config["join"]["max_nearest_km"]

basin_cols = basins.columns.drop("geometry").tolist()

# Nearest basin, measured in metres (EPSG:3035)
nearest = gpd.sjoin_nearest(
    outside[site_points.columns].to_crs(CRS_AREA),
    basins.to_crs(CRS_AREA),
    how="left",
    max_distance=MAX_KM * 1000,
    distance_col="distance_m",
).to_crs(CRS_GEO)

nearest[["campus_id", "country", "name", "pfaf_id", "distance_m"]]

,campus_id,country,name,pfaf_id,distance_m
0,existing_0,ES,Aire Canarias,NaN,NaN
317,existing_31,NL,Bluenap Americas,NaN,NaN
1073,existing_991,NO,R74n Vault,NaN,NaN
1074,existing_992,FR,Data,NaN,NaN


In [16]:
inside = joined[joined["pfaf_id"].notna()].assign(match_type="within", distance_m=0.0)
nearest["match_type"] = nearest["pfaf_id"].notna().map({True: "nearest", False: "no_match"})

site_basins = pd.concat([inside, nearest], ignore_index=True).drop(columns="index_right", errors="ignore")
print(site_basins["match_type"].value_counts())
print("Total:", len(site_basins))

match_type
within      1120
no_match       4
Name: count, dtype: int64
Total: 1124


In [17]:
b = config["scope"]["bbox"]
check = outside[["country", "name", "operators", "lat", "lon"]].copy()
check["inside_bbox"] = (
    check["lat"].between(b["south"], b["north"]) & check["lon"].between(b["west"], b["east"])
)
check

,country,name,operators,lat,lon,inside_bbox
0,ES,Aire Canarias,AIRE,28.118717,-15.429130,False
317,NL,Bluenap Americas,,12.177858,-68.955208,False
1073,NO,R74n Vault,,77.475486,23.475382,False
1074,FR,Data,ITH Center,-12.778454,45.225200,False


In [18]:
site_basins.loc[site_basins["match_type"] == "no_match", "match_type"] = "outside_scope"
site_basins["in_scope"] = site_basins["match_type"] != "outside_scope"

print(site_basins["match_type"].value_counts())
print("Sites in scope:", site_basins["in_scope"].sum())

match_type
within           1120
outside_scope       4
Name: count, dtype: int64
Sites in scope: 1120


In [19]:
s = site_basins[site_basins["in_scope"]]

labels = s["bws_label"].fillna("No data")
summary = pd.DataFrame({
    "sites": labels.value_counts(),
    "%": (labels.value_counts(normalize=True) * 100).round(1),
})
display(summary)

high = s["bws_cat"].isin([3, 4]).sum()
arid = s["bws_arid_low_use"].sum()
print(f"High or extremely high water stress: {high} sites ({high / len(s):.1%})")
print(f"Arid & low water use: {arid} sites")

,sites,%
bws_label,,
Low (<10%),407,36.3
High (40-80%),247,22.1
Low - Medium (10-20%),210,18.8
Medium - High (20-40%),154,13.8
Extremely High (>80%),102,9.1


High or extremely high water stress: 349 sites (31.2%)
Arid & low water use: 0 sites
